In [9]:
import pyodbc
import os

# Configurações do SQL Server
server = '192.168.88.91'           # ex: 'localhost\SQLEXPRESS'
database = 'SESC_SOPHIA_PSA'       # Nome da base
username = 'sa'                    # ex: 'sa'
password = 'Totvs@2025'            # ex: '123456'
folder_path = r'/media/andresen/MSATA2/Documentos/Trabalho/POLO DE DADOS/CLIENTES/SESC/CARGA.PRD/fotos.prd_psa'              # Caminho para salvar fotos

# Garante que a pasta existe
os.makedirs(folder_path, exist_ok=True)

# Conexão
conn_str = f"""
    DRIVER={{ODBC Driver 18 for SQL Server}};
    SERVER={server};
    DATABASE={database};
    UID={username};
    PWD={password};
    Encrypt=no;
    TrustServerCertificate=yes;
"""
conn = pyodbc.connect(conn_str)
cursor = conn.cursor()

# Consulta
query = """
SELECT 
    ZMIGRA_BASE.RA,
    sophia.DADOSPF_FOTO.IMG_FOTO,
    sophia.DADOSPF_FOTO.EXT
FROM sophia.DADOSPF_FOTO
JOIN ZMIGRA_BASE ON ZMIGRA_BASE.FISICA = sophia.DADOSPF_FOTO.FISICA
WHERE sophia.DADOSPF_FOTO.IMG_FOTO IS NOT NULL
"""

cursor.execute(query)

# Salvar os arquivos com nome baseado apenas no RA
for row in cursor.fetchall():
    ra = row.RA.strip() if row.RA else 'sem_ra'
    foto = row.IMG_FOTO
    extensao = row.EXT.strip().lower().replace('.', '')

    filename = f"{ra}.{extensao}"
    filepath = os.path.join(folder_path, filename)

    with open(filepath, 'wb') as f:
        f.write(foto)

    print(f"✔ Foto salva: {filepath}")

# Fecha a conexão
cursor.close()
conn.close()
